In [ ]:
#| default_exp coreml

# A fast decoder for Apple hardware

> Kokoro's decoder made exportable, split in two, and converted to Core ML: a drop-in `decode` for the Mac's GPU.

So far everything runs in PyTorch on the CPU. A Mac, an iPhone and an iPad all have a GPU and a Neural Engine that
Core ML can use. This notebook builds `CoreMLDecoder`, a replacement for notebook 100's `decode` that runs on them.

Only the decoder, because that's where the time goes:

In [ ]:
#| export
import copy
import os
import tempfile
import types

import coremltools as ct
import numpy as np
import torch
import torch.nn.functional as F
from coremltools.converters.mil import Builder as mb
from coremltools.converters.mil.frontend.torch.ops import _get_inputs, mul
from coremltools.converters.mil.frontend.torch.torch_op_registry import register_torch_op
from coremltools.converters.mil.mil import types as mil_types
from kokoro.custom_stft import CustomSTFT

from text_to_speech.core import DATA
from text_to_speech.voice import FRAME, decode, model

scikit-learn version 1.9.1 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
Torch version 2.14.0 has not been tested with coremltools. You may run into unexpected errors. Torch 2.7.0 is the most recent version that has been tested.
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torchao/quantization/quant_api.py:1558: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.
W1101 22:32:24.827000 58540 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W1101 22:32:24.864000 58540 .venv/lib/python3.13/site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is depr

In [ ]:
import time

import librosa
from fastcore.test import test_close, test_eq
from IPython.display import Audio
from kokoro.istftnet import TorchSTFT

from text_to_speech.voice import SR, analyse, say

In [ ]:
text = "Long paragraphs are split so pause and seek stay responsive, and the reader follows along word by word."
analyse(text)  # The first call loads the frontend
t = time.perf_counter()
u = analyse(text)
front = time.perf_counter() - t
t = time.perf_counter()
audio = decode(u)
print(
    f"{len(audio) / SR:.1f}s of audio: analyse {front * 1000:.0f} ms, decode {(time.perf_counter() - t) * 1000:.0f} ms"
)

7.0s of audio: analyse 89 ms, decode 797 ms


/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Around 90% of the time, and two thirds of the weights. The decoder is almost all one-dimensional convolutions,
exactly what GPUs and neural accelerators are for. Three things stop it converting as it is.

## 1. The STFT

The decoder's last layer predicts a tiny spectrogram, 11 frequencies every 5 samples, and an inverse STFT turns it
into audio. It also takes an STFT of its pitch-driven source on the way in. Kokoro uses `torch.stft`, which needs
complex numbers; converters don't handle those. Kokoro ships `CustomSTFT`, made of convolutions, for exactly this, but
it pads differently and its inverse is approximate (its comments say so):

In [ ]:
x = torch.randn(1, SR)
reference = TorchSTFT(20, 5, 20)
m0, p0 = reference.transform(x)
y0 = reference.inverse(m0, p0).squeeze()
custom = CustomSTFT(20, 5, 20)
m, p = custom.transform(x)
print(f"forward: largest error {((m * torch.exp(1j * p)) - (m0 * torch.exp(1j * p0))).abs().max():.2f}")
print(f"inverse: largest error {(custom.inverse(m0, p0).squeeze() - y0).abs().max():.2f}, on a signal of std 1")

forward: largest error 3.71
inverse: largest error 0.97, on a signal of std 1


The forward transform is right once it pads by reflection, as `torch.stft` does. The inverse misses two things: every
frequency except the first and last has a mirror image, so counts twice; and the overlapping windows don't add up to a
constant, so the result has to be divided by their sum. That's what `torch.istft` does, written as convolutions:

In [ ]:
#| export
class ExactSTFT(CustomSTFT):
    "Kokoro's convolutional STFT with torch.stft's reflect padding, and an inverse that matches torch.istft."

    def __init__(self, n_fft, hop):
        super().__init__(n_fft, hop, n_fft, pad_mode="reflect")
        k = np.arange(n_fft // 2 + 1)
        scale = np.where((k == 0) | (k == n_fft // 2), 1, 2) / n_fft  # Frequencies with a mirror image count twice
        angle = 2 * np.pi * np.outer(k, np.arange(n_fft)) / n_fft
        window = self.window.numpy()
        self.weight_backward_real = torch.tensor(np.cos(angle) * scale[:, None] * window, dtype=torch.float32)[:, None]
        self.weight_backward_imag = torch.tensor(np.sin(angle) * scale[:, None] * window, dtype=torch.float32)[:, None]
        self.register_buffer("window_squared", (self.window**2)[None, None])

    def inverse(self, magnitude, phase):
        real = F.conv_transpose1d(magnitude * torch.cos(phase), self.weight_backward_real, stride=self.hop_length)
        imag = F.conv_transpose1d(magnitude * torch.sin(phase), self.weight_backward_imag, stride=self.hop_length)
        envelope = F.conv_transpose1d(torch.ones_like(magnitude[:, :1]), self.window_squared, stride=self.hop_length)
        pad = self.n_fft // 2
        return ((real - imag) / envelope)[..., pad:-pad]

In [ ]:
exact = ExactSTFT(20, 5)
m, p = exact.transform(x)
test_close(m * torch.exp(1j * p), m0 * torch.exp(1j * p0), eps=1e-5)
test_close(exact.inverse(m0, p0).squeeze(), y0, eps=1e-5)
spectrum = (torch.rand(1, 11, 400) * 3, torch.randn(1, 11, 400))  # Not the STFT of anything, like the decoder's
test_close(exact.inverse(*spectrum), reference.inverse(*spectrum), eps=1e-5)

## 2. The sine source

The decoder's source turns the pitch curve into a phase by adding it up over the whole sentence, then takes the sine.
Two details break once it leaves PyTorch. Kokoro wraps each step's phase with `% 1`, and Core ML's converter gets that
wrong:

In [ ]:
#| export
def compile_coreml(m, path, units="CPU_ONLY"):
    "Save converted model `m` at `path` (an .mlpackage), compile it next to it, and load it on `units`."
    m.save(str(path))
    compiled = ct.utils.compile_model(str(path), str(path).replace(".mlpackage", ".mlmodelc"))
    return ct.models.CompiledMLModel(compiled, compute_units=getattr(ct.ComputeUnit, units))

In [ ]:
class Wrap(torch.nn.Module):
    def forward(self, x):
        return x % 1


x = torch.tensor([[0.25, 1.5, 0.001]])
traced = torch.jit.trace(Wrap(), (x,))
wrap = ct.convert(traced, inputs=[ct.TensorType(name="x", shape=x.shape)], minimum_deployment_target=ct.target.macOS15)
wrap = compile_coreml(wrap, os.path.join(tempfile.mkdtemp(), "wrap.mlpackage"))
print("PyTorch:", Wrap()(x).tolist(), " Core ML:", next(iter(wrap.predict({"x": x.numpy()}).values())).tolist())

PyTorch: [[0.25, 0.5, 0.0010000000474974513]]  Core ML: [[0.0, 0.0, 0.0]]


/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Model is not in eval mode. Consider calling '.eval()' on your model prior to conversion
Running MIL default pipeline:   0%|          | 0/95 [00:00<?, ? passes/s]/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '3', of the source model, has been renamed to 'var_3' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
Running MIL backend_mlprogram pipeline: 100%|##########| 12/12 [00:00<00:00, 72628.64 passes/s]


It treats `% 1` as whole numbers. Writing it `x - floor(x)` means the same for anything non-negative, and the
predicted pitch can dip a hair below zero in silence, so we clamp it at zero first.

The other detail is size. The phase reaches thousands of cycles over a sentence, which 32-bit floats hold well enough
but 16-bit floats, which the GPU and Neural Engine prefer, can't: they resolve numbers in the thousands only to the
nearest few. We'll keep this part in 32-bit. Taking the fractional part before the sine also keeps its argument small.

In [ ]:
#| export
def _f02sine(self, f0_values):
    "Kokoro's SineGen._f02sine, with F0 clamped at zero, % 1 written out, and the phase wrapped before the sine."
    rad_values = f0_values.clamp(min=0) / self.sampling_rate
    rad_values = rad_values - torch.floor(rad_values)
    rand_ini = torch.rand(f0_values.shape[0], f0_values.shape[2], device=f0_values.device)
    rand_ini[:, 0] = 0
    rad_values[:, 0, :] = rad_values[:, 0, :] + rand_ini
    rad_values = F.interpolate(rad_values.transpose(1, 2), scale_factor=1 / self.upsample_scale, mode="linear")
    cycles = torch.cumsum(rad_values.transpose(1, 2), dim=1)
    cycles = F.interpolate(
        cycles.transpose(1, 2) * self.upsample_scale, scale_factor=self.upsample_scale, mode="linear"
    )
    cycles = cycles.transpose(1, 2)
    return torch.sin(2 * torch.pi * (cycles - torch.floor(cycles)))

For a pitch curve that never goes below zero it's Kokoro's own, to within rounding. The rounding grows with the
harmonic, up to about 0.01 for the ninth: that's Kokoro taking the sine of numbers in the tens of thousands, which
wrapping first avoids.

In [ ]:
sinegen = model.decoder.generator.m_source.l_sin_gen
f0 = model.decoder.generator.f0_upsamp(u.F0.clamp(min=0)[:, None]).transpose(1, 2) * torch.arange(1, 10)
torch.manual_seed(0)
theirs = sinegen._f02sine(f0)
torch.manual_seed(0)
test_close(_f02sine(sinegen, f0), theirs, eps=2e-2)

## 3. Weight normalisation

Kokoro computes each convolution's weights from two tensors on every call (weight normalisation). Exporters want
constant weights, so we compute them once and fold them in.

`exportable_decoder` makes a copy of the decoder with all three changes.

In [ ]:
#| export
def exportable_decoder():
    "A copy of Kokoro's decoder that converters can handle, and that sounds the same."
    for m in model.decoder.modules():
        if hasattr(
            m, "weight_g"
        ):  # weight_norm caches a computed weight, which deepcopy refuses; it's recomputed anyway
            m.weight = m.weight.detach()
    dec = copy.deepcopy(model.decoder).eval()
    dec.generator.stft = ExactSTFT(dec.generator.stft.filter_length, dec.generator.stft.hop_length)
    sinegen = dec.generator.m_source.l_sin_gen
    sinegen._f02sine = types.MethodType(_f02sine, sinegen)
    for m in dec.modules():
        if hasattr(m, "weight_g"):
            torch.nn.utils.remove_weight_norm(m)
    return dec.requires_grad_(False)  # After folding, which makes new parameters

The decoder adds random noise, so two runs never match exactly. The fair test is that ours is about as close to
Kokoro's as two of Kokoro's runs with different seeds are to each other, measured between log mel spectrograms:

In [ ]:
def distance(a, b):
    "Mean difference between the log mel spectrograms of two clips."
    mel = lambda y: np.log(np.maximum(librosa.feature.melspectrogram(y=y, sr=SR, n_mels=80), 1e-5))
    n = min(len(a), len(b))
    return np.abs(mel(a[:n]) - mel(b[:n])).mean()


def seeded(decoder, seed=0):
    torch.manual_seed(seed)
    return decoder(u)


dec = exportable_decoder()
ours = seeded(lambda u: dec(u.asr, u.F0, u.N, u.style).squeeze().numpy())
floor = distance(seeded(decode, 0), seeded(decode, 1))
print(f"two seeds differ by {floor:.3f}; ours differs from Kokoro by {distance(ours, seeded(decode)):.3f}")
assert distance(ours, seeded(decode)) < 3 * floor
Audio(ours, rate=SR)

two seeds differ by 0.073; ours differs from Kokoro by 0.160


## Split in two

The part that needs 32 bits, the source and its STFT, is a sliver of the work. So the decoder becomes two models:
`HarmonicSource`, pitch to the source's spectrum, which stays 32-bit on the CPU, and `Body`, everything else, which
can be 16-bit on the GPU. `Body` is Kokoro's own `Decoder.forward` and `Generator.forward` with the source as an
input.

In [ ]:
#| export
class HarmonicSource(torch.nn.Module):
    "Pitch -> the STFT of the decoder's sine source. Adds up phase over the whole sentence, so needs 32 bits."

    def __init__(self, dec):
        super().__init__()
        self.gen = dec.generator

    def forward(self, F0_curve):
        f0 = self.gen.f0_upsamp(F0_curve[:, None]).transpose(1, 2)
        har_source, _, _ = self.gen.m_source(f0)
        spec, phase = self.gen.stft.transform(har_source.transpose(1, 2).squeeze(1))
        return torch.cat([spec, phase], dim=1)


class Body(torch.nn.Module):
    "The rest of the decoder: Kokoro's Decoder.forward and Generator.forward, taking the source as an input."

    def __init__(self, dec):
        super().__init__()
        self.dec = dec

    def forward(self, asr, F0_curve, N, s, har):
        d, g = self.dec, self.dec.generator
        F0, N = d.F0_conv(F0_curve.unsqueeze(1)), d.N_conv(N.unsqueeze(1))
        x = d.encode(torch.cat([asr, F0, N], axis=1), s)
        asr_res, res = d.asr_res(asr), True
        for block in d.decode:
            if res:
                x = torch.cat([x, asr_res, F0, N], axis=1)
            x = block(x, s)
            res = res and block.upsample_type == "none"
        for i in range(g.num_upsamples):
            x = F.leaky_relu(x, negative_slope=0.1)
            x_source = g.noise_res[i](g.noise_convs[i](har), s)
            x = g.ups[i](x)
            if i == g.num_upsamples - 1:
                x = g.reflection_pad(x)
            x = x + x_source
            x = sum(g.resblocks[i * g.num_kernels + j](x, s) for j in range(g.num_kernels)) / g.num_kernels
        x = g.conv_post(F.leaky_relu(x))
        spec, phase = torch.exp(x[:, : g.post_n_fft // 2 + 1, :]), torch.sin(x[:, g.post_n_fft // 2 + 1 :, :])
        return g.stft.inverse(spec, phase)

With the same seed, the two halves are the whole:

In [ ]:
source, body = HarmonicSource(dec).eval(), Body(dec).eval()
with torch.no_grad():
    split = seeded(lambda u: body(u.asr, u.F0, u.N, u.style, source(u.F0)).squeeze().numpy())
test_close(split, ours, eps=1e-5)

## To Core ML

Core ML wants fixed shapes, so each half is converted at three sizes, 128, 256 and 512 frames (3.2, 6.4 and 12.8 s),
and each sentence is padded up to the next. Two more converter gaps: Kokoro takes `rsqrt` of the integer 2, which
PyTorch quietly makes a float and coremltools refuses, and it spells `*` as `torch.multiply`.

In [ ]:
#| export
@register_torch_op(override=True)
def rsqrt(context, node):
    "Kokoro takes rsqrt of the integer 2, which PyTorch promotes to a float and coremltools doesn't."
    x = _get_inputs(context, node, expected=1)[0]
    if mil_types.is_int(x.dtype):
        x = mb.cast(x=x, dtype="fp32")
    context.add(mb.rsqrt(x=x, name=node.name))


@register_torch_op
def multiply(context, node):
    "torch.multiply is another name for `*`."
    mul(context, node)


def to_coreml(module, args, names, precision=ct.precision.FLOAT16):
    "Trace `module` at the shapes of `args` and convert it."
    traced = torch.jit.trace(module, args, check_trace=False)
    inputs = [ct.TensorType(name=n, shape=a.shape) for n, a in zip(names, args)]
    return ct.convert(
        traced,
        inputs=inputs,
        outputs=[ct.TensorType(name="out")],
        compute_precision=precision,
        minimum_deployment_target=ct.target.macOS15,
    )

## Padding

Fixed sizes mean padding, and padding is not as innocent as it looks. The decoder normalises every channel over the
whole time axis (instance norm, in each AdaIN layer), so zeros on the end change the statistics of the real part
too. The padded part of the output is thrown away, so we can fill it with anything: repeating the sentence keeps the
statistics close to the sentence's own. In PyTorch, padding this 278-frame sentence to 512:

In [ ]:
#| export
def tile(x, n):
    "Repeat `x` along its last axis to length `n`: padding whose statistics look like the sentence's."
    return torch.cat([x] * -(-n // x.shape[-1]), -1)[..., :n]

In [ ]:
T, B = u.asr.shape[-1], 512
zeros = lambda x, n: F.pad(x, (0, n - x.shape[-1]))
for name, pad in [("zeros", zeros), ("the sentence again", tile)]:
    with torch.no_grad():
        torch.manual_seed(0)
        F0 = pad(u.F0, 2 * B)
        y = body(pad(u.asr, B), F0, pad(u.N, 2 * B), u.style, source(F0)).squeeze().numpy()[: T * FRAME]
    print(f"padded with {name:<18} distance from Kokoro {distance(y, seeded(decode)):.3f}")
print(f"not padded {'':<17} distance from Kokoro {distance(split, seeded(decode)):.3f}")

padded with zeros              distance from Kokoro 0.815
padded with the sentence again distance from Kokoro 0.375
not padded                   distance from Kokoro 0.160


`CoreMLDecoder` converts the body at each size the first time and keeps them in `data/coreml`, so later it only loads
them. It always uses compiled models: predicting with a model straight out of `ct.convert` crashed later work in the
same process.

The source stays in PyTorch, where it takes about 3 ms per second of audio. It converts fine, but on this Mac running
the converted source broke PyTorch's next decoder run in the same process, with a crash deep inside `torch.sin`. On a
phone there's no PyTorch to collide with, so this is a Mac-only workaround. Sentences longer than the largest size
fall back to PyTorch altogether.

In [ ]:
#| export
BUCKETS = [128, 256, 512]  # Frames: 3.2, 6.4 and 12.8 s
COREML = DATA / "coreml"


@torch.no_grad()
def _build(T, dec):
    "Convert the body of `dec` for T frames to 16-bit Core ML in COREML, unless it's there already."
    COREML.mkdir(parents=True, exist_ok=True)
    if not (COREML / f"body{T}.mlmodelc").exists():
        F0 = torch.zeros(1, 2 * T)
        args = (torch.zeros(1, 512, T), F0, torch.zeros(1, 2 * T), torch.zeros(1, 128), HarmonicSource(dec)(F0))
        compile_coreml(to_coreml(Body(dec), args, ["asr", "F0", "N", "s", "har"]), COREML / f"body{T}.mlpackage")


class CoreMLDecoder:
    "A drop-in for `decode`: the source in 32-bit PyTorch, the body in 16-bit Core ML on `units`."

    def __init__(self, units="CPU_AND_GPU"):
        dec = exportable_decoder()
        self.source = HarmonicSource(dec).eval()
        for T in BUCKETS:
            _build(T, dec)
        units = getattr(ct.ComputeUnit, units)
        self.body = {
            T: ct.models.CompiledMLModel(str(COREML / f"body{T}.mlmodelc"), compute_units=units) for T in BUCKETS
        }

    @torch.no_grad()
    def __call__(self, u):
        T = u.asr.shape[-1]
        if T > BUCKETS[-1]:
            return decode(u)
        B = next(b for b in BUCKETS if b >= T)
        F0 = tile(u.F0, 2 * B)
        inputs = {"asr": tile(u.asr, B), "F0": F0, "N": tile(u.N, 2 * B), "s": u.style, "har": self.source(F0)}
        return self.body[B].predict({k: v.numpy() for k, v in inputs.items()})["out"].squeeze()[: T * FRAME]

In [ ]:
coreml = CoreMLDecoder()
fast = seeded(coreml)
test_eq(len(fast), len(audio))
with torch.no_grad():
    torch.manual_seed(0)
    F0 = tile(u.F0, 2 * B)
    same_input = body(tile(u.asr, B), F0, tile(u.N, 2 * B), u.style, source(F0)).squeeze().numpy()[: T * FRAME]
print(f"Core ML vs PyTorch on the same input: {distance(fast, same_input):.3f}. Two seeds: {floor:.3f}")
print(f"Core ML vs Kokoro: {distance(fast, seeded(decode)):.3f}")
assert distance(fast, same_input) < floor


def realtime(decoder, reps=5):
    "How many times faster than real time `decoder` renders `u`."
    decoder(u)
    t = time.perf_counter()
    for _ in range(reps):
        decoder(u)
    return len(audio) / SR / ((time.perf_counter() - t) / reps)


print(f"PyTorch on the CPU: {realtime(decode):.0f}x real time. Core ML on the GPU: {realtime(coreml):.0f}x")
Audio(fast, rate=SR)

Core ML vs PyTorch on the same input: 0.011. Two seeds: 0.073
Core ML vs Kokoro: 0.374
PyTorch on the CPU: 10x real time. Core ML on the GPU: 27x


It's a drop-in, so `say` takes it as it is:

In [ ]:
audio, ws = say("Same voice, same words, on the GPU.", decoder=coreml)
Audio(audio, rate=SR)

## Summary

- **Three changes** make Kokoro's decoder convertible: an exact convolutional STFT, a sine source written so the
  converter can't get `%` wrong, and weight normalisation folded into the weights.
- **Split in two**, the part that must be 32-bit stays in PyTorch on the CPU and the rest runs in 16-bit on the GPU.
- **`CoreMLDecoder`** is a drop-in for `decode`.

The conversion itself is closer than two seeds. What remains of the distance from Kokoro is the padding the fixed
sizes need; the appendix measures its effect on predicted MOS (none), along with the Neural Engine, which only takes
inputs of 1.6 s or less, and 8-, 6- and 4-bit weights. Notebook 104 does the same for Android, and notebook
105 puts this decoder in the player.